In [78]:
import sys
import os
import json
from glob import glob
import shutil

In [79]:
def read_jsonl(file_path):
    with open(file_path, "r") as file:
        return [json.loads(line) for line in file.readlines()]

In [131]:
def extract_raw_cheatsheet(generator_prompt):
    """
    Extracts the cheatsheet from a given generator prompt
    """
    g = generator_prompt.split("-----")
    cheatsheet_section = [section for section in g if "CHEATSHEET:" in section]
    
    if (len(cheatsheet_section) != 1):
        raise Exception("Invalid cheatsheet format!!")
    
    cheatsheet_section = cheatsheet_section[0]
    
    DELIM = "'''\n"
    first_str = cheatsheet_section.find(DELIM)
    second_str = cheatsheet_section.find(DELIM, first_str + 1)
    
    return cheatsheet_section[first_str + len(DELIM): second_str - 1]

def extract_memory_items(cheatsheet_raw):
    MEMORY_DELIM = "<memory_item>"
    MEMORY_END_DELIM = "</memory_item>"
    items = cheatsheet_raw.split(MEMORY_END_DELIM)
    
    items = [item[len(MEMORY_DELIM) + 1:] for item in items if MEMORY_DELIM in item]
    
    return items

def mem_to_str(memory_item: dict) -> str:
    
    if 'unique_id' in memory_item:
        return f"""
ID #{memory_item['unique_id']}:
Strategy:
{memory_item['strategy']}
        """
    
    return f"""
ID #{memory_item['id']}:
{memory_item['text']}
        """

In [132]:
# TASK_LIST = ["AIME_2024", "AIME_2025", "AIME_2020_2024", "GameOf24", "GPQA_Diamond", "MMLU_Pro_Engineering", "MMLU_Pro_Physics", "MathEquationBalancer", "IneqMath/openai"]

TASK_LIST = ["IneqMath"]

results = {}

for TASK in TASK_LIST:
    all_files = glob(f"results/{TASK}/**/**/*.jsonl", recursive=True)
    all_files = list(set(all_files))
    all_files.sort()
    
    KEYWORDS = ["Strategic", "JSON_Memory"]
    
    files = []
    
    for keyword in KEYWORDS:
        files += [file for file in all_files if keyword in file]
    
    if len(files) == 0:
        continue
    
    questions_dict = {}
    
    print(f"{TASK} ==============================================")
    for file in files:
        print(file)
        print(d.keys())
        data = read_jsonl(file)

        for d in data:
            # print(d['steps'][0]['generator_prompt'])
            print("===============================")
            question = d['input']
            print(question)
            
            memory_items = []
            if 'retrieved_items' in d['steps'][0]:
                print(d['steps'][0]['retrieved_items'])
                memory_items = [mem_to_str(mem) for mem in d['steps'][0]['retrieved_items']]
                # print(memory_items)
            else:
                generator_prompt = d['steps'][0]['generator_prompt']
                cheatsheet_raw = extract_raw_cheatsheet(generator_prompt)
                memory_items = extract_memory_items(cheatsheet_raw)
            print(len(memory_items))
            
            if question not in questions_dict:
                questions_dict[question] = {}
                
            questions_dict[question][file] = memory_items
            
            print("===============================")
    print('--------------------------------')
    results[TASK] = questions_dict

IneqMath ==============================================
results/IneqMath/gpt-4o_DynamicCheatsheet_StrategicChunkRetrieval_prob0.8.jsonl
dict_keys(['input', 'target', 'raw_input', 'input_txt', 'steps', 'final_answer', 'final_output', 'final_cheatsheet', 'memory_store_text', 'memory_store_size', 'data_id', 'problem_type'])
Question #1:
Let $a, b, c$ be positive numbers such that $a^2 + b^2 + c^2 = 3$. Consider the following inequality:
$$
\frac{b+c}{a^2} + \frac{c+a}{b^2} + \frac{a+b}{c^2} \quad () \quad 3\left(\frac{1}{a} + \frac{1}{b} + \frac{1}{c} - a b c\right).
$$

Determine the correct inequality relation to fill in the blank.

Options:

(A) $\leq$ 

(B) $\geq$

(C) $=$ 

(D) $<$

(E) $>$

(F) None of the above

Choices:
(A) $\leq$
(B) $\geq$
(C) $=$
(D) $<$
(E) $>$
(F) None of the above

(Select the correct relation from the choices above. State your final answer as the choice letter, e.g. (A).)
[]
0
Question #2:
Let $a, b, c > 0$. Find the smallest constant $C$ such that the foll

In [141]:
aime = results['IneqMath']
i = 0
for question, mem_dict in aime.items():
    i += 1
    if i % 25 != 0:
        continue
    print(f"{question.split(':')[0]}  ===================")
    for method, memories in mem_dict.items():
        method_parts = method.split("_")
        print(f"{method_parts[-2]} {method_parts[-1]} ({len(memories)}) ---------------")
        for m in memories:
            print(m)

Question #25  ===================
StrategicChunkRetrieval prob0.8.jsonl (2) ---------------

ID #0:
<memory_item>
<description>
Using symmetry and specific case testing to solve inequalities involving symmetric constraints. This involves recognizing patterns, testing specific symmetric cases, and comparing results to known constants. (Reference: Q24)
</description>
<example>
**Strategy**:
1. **Identify Symmetry**: Recognize symmetry in the problem constraints, such as \(xy + yz + zx = 1\).
2. **Test Specific Symmetric Cases**: Consider cases where variables are equal, e.g., \(x = y = z\), to simplify the problem.
3. **Calculate and Compare**: Compute the expression for the symmetric case and compare it to the given constant.
4. **Use Known Values**: Compare square roots or other constants to determine the inequality relation.

**Example Application**:
For the inequality \(\frac{1}{x+x^3} + \frac{1}{y+y^3} + \frac{1}{z+z^3} \quad () \quad \frac{9 \sqrt{2}}{4}\) with \(xy + yz + zx = 1\)